In [ ]:
# Install Tiny-SIMA into the active Jupyter kernel.
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.environ.setdefault(
    "XLA_PYTHON_CLIENT_MEM_FRACTION",
    os.environ.get("TINY_SIMA_JAX_MEMORY_FRACTION", "0.60"),
)

if not ((3, 11) <= sys.version_info[:2] < (3, 13)):
    raise RuntimeError("Tiny-SIMA requires Python 3.11 or 3.12.")

repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Open this notebook from the Tiny-SIMA repository.")

cuda_override = os.environ.get("TINY_SIMA_USE_CUDA")
if cuda_override is None:
    has_cuda = (
        shutil.which("nvidia-smi") is not None
        and subprocess.run(
            ["nvidia-smi"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            check=False,
        ).returncode == 0
    )
else:
    has_cuda = cuda_override.lower() in {"1", "true", "yes"}

extras = "notebook,cuda12" if has_cuda else "notebook"
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--quiet",
    "--no-cache-dir",
    "--disable-pip-version-check",
    "-e",
    f"{repo_root}[{extras}]",
])

source_root = str(repo_root / "src")
if source_root not in sys.path:
    sys.path.insert(0, source_root)

print("Repository:", repo_root)
print("Source path added to kernel:", source_root)
print("CUDA detected:", has_cuda)


# Phase 4, executor: widen the training distribution

Notebook 06 left the executor as the ceiling. With the planner within two boundaries of the 45 achievable, every closed-loop condition, the scripted oracle included, ended most failures in a `collect_wood` timeout, and plan order alone moved the oracle from 0.250 to 0.609 (McNemar p < 0.001). The default order places the table after the first wood, and every later `collect_wood` then starts in a world the executor never trained on. `local_navigation` always starts at the map centre in a cleared window, with a single target, a reserved straight route, the minimal inventory and the clock at dawn. A chain boundary differs in every one of those respects.

The notebook runs four steps, each against the same paired worlds:

0. **Executor-only benchmark** of the current checkpoint. It covers both fixed plans over the same 512 worlds, success at every chain boundary by skill and distance band, and the in-distribution `local_navigation` check.
1. **Counterfactual probes.** Each recorded boundary is replayed with one training-distribution difference undone: tables removed, view cleared, training inventory, training vitals, dawn clock. Each edit is paired against the unedited replay of the same boundaries, so it reads how much that difference costs. The unedited replay must reproduce the chain exactly (`replay_agreement == 1.0`), because a greedy executor in a mob-free world is a function of its starting state.
2. **Training** on the `randomized` scenario. Each knob in `ScenarioRandomization` undoes one of those differences and can be switched off as an ablation. `local_navigation` at the same budget, horizon and distances is the ablation that separates the scenario from the budget.
3. **Paired old-against-new comparison** on the benchmark worlds, plus the regression check on `local_navigation`.

The success criterion is closing the gap between the default order and collect-first with the stochastic executor, the setting in which notebook 06 measured 0.250 against 0.609. Because both plans run on the same worlds, the gap is read episode by episode.

Run the `smoke` preset first: it trains the debug configuration and exercises every cell in minutes. `full` is the reported run.

In [ ]:
import json
import time

import jax
import matplotlib.pyplot as plt
import numpy as np

from tiny_sima.config import load_skill_config
from tiny_sima.envs import scenario_randomization_from_config
from tiny_sima.evaluation import (
    COLLECT_FIRST_PLAN,
    COUNTERFACTUAL_EDITS,
    EXECUTOR_BENCHMARK_SEED,
    WOOD_TO_STONE_CHAIN,
    benchmark_executor,
    compare_executor_benchmarks,
    compare_paired_successes,
    evaluate_goal_conditioned_policy,
    failed_rollout_features,
    json_ready_benchmark,
    learned_skill_executor_from_checkpoint,
    probe_counterfactuals,
    success_table,
    summarize_rollout_features,
)
from tiny_sima.executor.skill_ppo import train_goal_conditioned

print("JAX devices:", jax.devices())
print("Default plan:", [skill.value for skill in WOOD_TO_STONE_CHAIN.reference_skills])
print("Collect-first plan:", [skill.value for skill in COLLECT_FIRST_PLAN])
print("Counterfactual edits:", [edit.name for edit in COUNTERFACTUAL_EDITS])

In [ ]:
# The checkpoint notebook 06 measured. It is the reference every new run is
# paired against.
BASELINE_CHECKPOINT = (
    repo_root / "runs" / "skill-ppo-local-1m" / "checkpoints" / "final"
)

PRESET = "full"
PRESETS = {
    "smoke": {
        "config": "skill_ppo_randomized_debug.yaml",
        "benchmark_episodes": 8,
        "local_episodes": 8,
        "randomized_episodes": 8,
        "train_seeds": (0,),
        "ablation_seeds": (0,),
    },
    "full": {
        "config": "skill_ppo_randomized_10m.yaml",
        # Chains run in lockstep, so episodes cost batch width, not wall clock;
        # 512 puts the standard error of a chain success rate near 0.02.
        "benchmark_episodes": 512,
        "local_episodes": 256,
        "randomized_episodes": 256,
        "train_seeds": (0, 1, 2),
        # One budget-matched local_navigation run separates the scenario from
        # the ten-fold budget and the longer horizon. Set to () to skip it.
        "ablation_seeds": (0,),
    },
}
if PRESET not in PRESETS:
    raise ValueError(f"PRESET must be one of {sorted(PRESETS)}.")
preset = PRESETS[PRESET]

# Steps can be switched off when resuming. Training skips any run whose
# summary.json already exists unless RETRAIN is set.
RUN_COUNTERFACTUALS = True
RUN_TRAINING = True
RETRAIN = False

# Seeds. The benchmark worlds are disjoint from notebook 06's calibration
# (7, 23) and evaluation (51) seeds, so selecting a checkpoint here does not
# leak into the numbers that notebook reports. The randomized evaluation seed
# is disjoint from the training seeds.
BENCHMARK_SEED = EXECUTOR_BENCHMARK_SEED
RANDOMIZED_EVAL_SEED = 202
CHAIN_HORIZON = 64
# The horizon local_navigation was trained at, for the regression check.
LOCAL_HORIZON = 32

OUTPUT_DIR = repo_root / "runs" / f"executor-improvement-{PRESET}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

base_config = load_skill_config(repo_root / "configs" / preset["config"])
randomization = scenario_randomization_from_config(base_config)
RUNS = {
    **{
        f"randomized-seed{seed}": {"seed": seed}
        for seed in preset["train_seeds"]
    },
    **{
        f"local-matched-seed{seed}": {
            "seed": seed,
            "skill_scenario": "local_navigation",
        }
        for seed in preset["ablation_seeds"]
    },
}

print("Baseline exists:", BASELINE_CHECKPOINT.exists())
print("Preset:", PRESET, preset)
print("Training config:", preset["config"], "updates:", base_config.num_updates)
print("Randomization:", randomization)
print("Runs:", list(RUNS))

In [ ]:
def save_json(name, payload):
    path = OUTPUT_DIR / f"{name}.json"
    path.write_text(json.dumps(payload, indent=2, sort_keys=True, default=str))
    return path


def band_rows(table):
    """Flatten a success table keyed by (skill, band) for printing and JSON."""

    return [
        {"skill": skill, "band": band, **cell}
        for (skill, band), cell in table.items()
    ]


def print_chains(name, benchmark):
    rates = {
        plan: result["success_rate"] for plan, result in benchmark["chains"].items()
    }
    paired = benchmark["plan_comparisons"]["default_vs_collect_first"]
    print(
        f"{name:24s} default={rates['default']:.3f} "
        f"collect_first={rates['collect_first']:.3f} "
        f"gap={rates['collect_first'] - rates['default']:+.3f} "
        f"discordant={paired['reference_only']}/{paired['candidate_only']} "
        f"p={paired['p_value']:.3g}"
    )


def print_bands(benchmark, plan):
    table = success_table(
        benchmark["boundaries"][plan],
        lambda boundary: (boundary.skill_id.value, boundary.band),
    )
    print(f"{'skill':22s} {'band':>11s} {'n':>5s} {'success':>8s}")
    for row in band_rows(table):
        print(
            f"{row['skill']:22s} {row['band']:>11s} "
            f"{row['count']:5d} {row['success_rate']:8.3f}"
        )

## Step 0: benchmark the current executor

The greedy executor makes every option a function of its starting state, which the counterfactual replay below depends on. The boundary tables count first attempts only: a retry starts in a world the first attempt changed, so it would mix the executor's competence with its own earlier failure.

Read `discordant` as (default only, collect-first only) episodes. The gap is the quantity this phase is trying to close.

In [ ]:
if not BASELINE_CHECKPOINT.exists():
    raise FileNotFoundError(f"Missing baseline checkpoint: {BASELINE_CHECKPOINT}")

executors = {"baseline": learned_skill_executor_from_checkpoint(BASELINE_CHECKPOINT)}
benchmarks = {}

started = time.perf_counter()
benchmarks["baseline"] = benchmark_executor(
    executors["baseline"],
    num_episodes=preset["benchmark_episodes"],
    seed=BENCHMARK_SEED,
    skill_horizon=CHAIN_HORIZON,
    local_episodes_per_skill=preset["local_episodes"],
    local_horizon=LOCAL_HORIZON,
)
print(f"seconds={time.perf_counter() - started:.1f}")
save_json("benchmark-baseline", json_ready_benchmark(benchmarks["baseline"]))

print_chains("baseline", benchmarks["baseline"])
print("\nlocal_navigation regression check:")
for result in benchmarks["baseline"]["local_navigation"]:
    print(f"  {result['skill_id']:22s} success={result['success_rate']:.3f}")

In [ ]:
# Success at every first attempt, by skill and in-view distance band. The
# default plan's later collect_wood rows are the ones the gap lives in.
for plan in benchmarks["baseline"]["chains"]:
    print(f"\n=== {plan} ===")
    print_bands(benchmarks["baseline"], plan)

# The same boundaries split by how often the skill had been requested, which
# separates the first collect_wood, in the untouched world, from the later ones.
default_boundaries = benchmarks["baseline"]["boundaries"]["default"]
occurrence_table = success_table(
    default_boundaries,
    lambda boundary: (
        boundary.skill_id.value,
        boundary.occurrence,
        boundary.table_in_view,
    ),
)
print(f"\n{'skill':22s} {'occurrence':>10s} {'table':>6s} {'n':>5s} {'success':>8s}")
for (skill, occurrence, table), cell in occurrence_table.items():
    print(
        f"{skill:22s} {occurrence:10d} {str(table):>6s} "
        f"{cell['count']:5d} {cell['success_rate']:8.3f}"
    )

## Step 1: which training difference costs the most

Every edit is applied to the recorded boundary state and the option is replayed from there. Skills an edit would make impossible are left out of that comparison: crafting needs its table, and only the skills that use them have their inventory reset. `all` composes every edit. `cand_only` counts boundaries the edit rescued and `ref_only` boundaries it broke.

Confound control: an edit that helps only when it also shortens the path to the target is a reachability effect rather than a distribution one. That is why the distance band is printed for every rescued boundary.

The failed-rollout features describe what the executor did when it failed. `ever_near` without `interact_while_facing` means it reached the target and did not use it. A low `ever_near` with many `blocked_moves` means it never found a way around an obstacle. A high `revisit_fraction` means it was pacing.

In [ ]:
if RUN_COUNTERFACTUALS:
    probe_boundaries = [
        boundary
        for boundaries in benchmarks["baseline"]["boundaries"].values()
        for boundary in boundaries
    ]
    started = time.perf_counter()
    counterfactuals = probe_counterfactuals(
        executors["baseline"],
        probe_boundaries,
        skill_horizon=CHAIN_HORIZON,
    )
    print(
        f"boundaries={counterfactuals['num_boundaries']} "
        f"replay_agreement={counterfactuals['replay_agreement']:.3f} "
        f"seconds={time.perf_counter() - started:.1f}"
    )
    if counterfactuals["replay_agreement"] < 1.0:
        print(
            "WARNING: the unedited replay disagrees with the chain, so edits "
            "are not isolated. Check that the executor is greedy."
        )

    print(
        f"\n{'edit':20s} {'n':>5s} {'both':>5s} {'ref_only':>8s} "
        f"{'cand_only':>9s} {'diff':>7s} {'p':>7s}"
    )
    for name, comparison in counterfactuals["comparisons"].items():
        print(
            f"{name:20s} {comparison['num_pairs']:5d} {comparison['both']:5d} "
            f"{comparison['reference_only']:8d} {comparison['candidate_only']:9d} "
            f"{comparison['difference']:+7.3f} {comparison['p_value']:7.3g}"
        )

    # Per skill, because an edit that only matters for collect_wood is
    # diluted by every crafting boundary it does nothing to.
    print(f"\n{'edit':20s} {'skill':22s} {'n':>5s} {'before':>7s} {'after':>7s}")
    for name, traces in counterfactuals["traces"].items():
        if name == "none":
            continue
        pairs = [
            (boundary, original, edited)
            for boundary, original, edited in zip(
                probe_boundaries,
                counterfactuals["traces"]["none"],
                traces,
                strict=True,
            )
            if edited is not None
        ]
        for skill in sorted({boundary.skill_id.value for boundary, _, _ in pairs}):
            selected = [pair for pair in pairs if pair[0].skill_id.value == skill]
            before = np.mean([original.probe.succeeded for _, original, _ in selected])
            after = np.mean([edited.probe.succeeded for _, _, edited in selected])
            print(
                f"{name:20s} {skill:22s} {len(selected):5d} "
                f"{before:7.3f} {after:7.3f}"
            )

    # Confound control: the bands of the boundaries each edit rescued.
    print("\nrescued boundaries by band:")
    for name, traces in counterfactuals["traces"].items():
        if name == "none":
            continue
        rescued = [
            boundary.band
            for boundary, original, edited in zip(
                probe_boundaries,
                counterfactuals["traces"]["none"],
                traces,
                strict=True,
            )
            if edited is not None
            and edited.probe.succeeded
            and not original.probe.succeeded
        ]
        counts = {band: rescued.count(band) for band in sorted(set(rescued))}
        print(f"  {name:20s} {counts}")

    save_json(
        "counterfactuals-baseline",
        {
            "num_boundaries": counterfactuals["num_boundaries"],
            "replay_agreement": counterfactuals["replay_agreement"],
            "comparisons": counterfactuals["comparisons"],
        },
    )

In [ ]:
if RUN_COUNTERFACTUALS:
    feature_rows = failed_rollout_features(
        probe_boundaries,
        counterfactuals["traces"]["none"],
    )
    print(f"failed first attempts: {len(feature_rows)}")
    for skill in sorted({row["skill_id"] for row in feature_rows}):
        rows = [row for row in feature_rows if row["skill_id"] == skill]
        summary = summarize_rollout_features(rows)
        print(f"\n{skill}")
        for key, value in summary.items():
            shown = f"{value:.3f}" if isinstance(value, float) else value
            print(f"  {key:24s} {shown}")
    save_json("failed-rollouts-baseline", feature_rows)

## Step 2: train on the randomized scenario

Every run starts from the same configuration and differs only in seed, except the `local-matched` ablation, which switches the scenario back to `local_navigation` at the same budget, horizon and distance range. Training logs every `log_every` updates. A completed run is reloaded from its `summary.json`, so this cell can be rerun after an interruption without retraining what already finished.

Training-time success is measured on the training distribution and is not comparable across scenarios. The evaluations below are.

In [ ]:
summaries = {}
for name, overrides in RUNS.items():
    run_dir = OUTPUT_DIR / name
    summary_path = run_dir / "summary.json"
    if summary_path.is_file() and not RETRAIN:
        summaries[name] = json.loads(summary_path.read_text())
        print(f"{name}: loaded {summary_path}")
        continue
    if not RUN_TRAINING:
        print(f"{name}: not trained and RUN_TRAINING is False; skipped")
        continue
    config = base_config.with_overrides(output_dir=str(run_dir), **overrides)
    started = time.perf_counter()
    summaries[name] = train_goal_conditioned(config)
    print(
        f"{name}: seconds={time.perf_counter() - started:.0f} "
        f"train_macro_success={summaries[name]['macro_success_rate']:.3f}"
    )

for name, summary in summaries.items():
    per_skill = " ".join(
        f"{skill}={metrics['success_rate']:.2f}"
        for skill, metrics in summary["per_skill"].items()
        if metrics["success_rate"] is not None
    )
    print(f"{name:24s} {summary['environment']:36s} {per_skill}")
    executors[name] = learned_skill_executor_from_checkpoint(summary["checkpoint"])

In [ ]:
# Held-out randomized worlds: in distribution for the new runs, out of it for
# the baseline. A run that does not beat the baseline here has not learned the
# scenario, and nothing downstream is worth reading.
randomized_results = {
    name: evaluate_goal_conditioned_policy(
        executor.params,
        executor.config,
        num_episodes_per_skill=preset["randomized_episodes"],
        seed=RANDOMIZED_EVAL_SEED,
        scenario="randomized",
        horizon=CHAIN_HORIZON,
        min_distance=base_config.skill_min_distance,
        max_distance=base_config.skill_max_distance,
        distractor_count=base_config.skill_distractor_count,
        randomization=randomization,
    )
    for name, executor in executors.items()
}
skill_names = [result["skill_id"] for result in randomized_results["baseline"]]
print(f"{'skill':22s} " + " ".join(f"{name:>24s}" for name in randomized_results))
for index, skill in enumerate(skill_names):
    print(
        f"{skill:22s} "
        + " ".join(
            f"{results[index]['success_rate']:24.3f}"
            for results in randomized_results.values()
        )
    )
save_json("randomized-eval", randomized_results)

## Step 3: paired comparison against the baseline

Each run is benchmarked on the same worlds as the baseline. Plan success pairs episode by episode against the baseline. Boundary bands cannot be paired, because a different executor reaches different states, so they are shown side by side. The `local_navigation` check must not regress: a run that gains on chains by forgetting the original skills is not an improvement.

In [ ]:
comparisons = {}
for name in summaries:
    started = time.perf_counter()
    benchmarks[name] = benchmark_executor(
        executors[name],
        num_episodes=preset["benchmark_episodes"],
        seed=BENCHMARK_SEED,
        skill_horizon=CHAIN_HORIZON,
        local_episodes_per_skill=preset["local_episodes"],
        local_horizon=LOCAL_HORIZON,
    )
    comparisons[name] = compare_executor_benchmarks(
        benchmarks["baseline"],
        benchmarks[name],
    )
    save_json(f"benchmark-{name}", json_ready_benchmark(benchmarks[name]))
    print(f"{name}: seconds={time.perf_counter() - started:.1f}")

print()
for name, benchmark in benchmarks.items():
    print_chains(name, benchmark)

print(
    f"\n{'run':24s} {'plan':14s} {'both':>5s} {'base_only':>9s} "
    f"{'run_only':>8s} {'diff':>7s} {'p':>7s}"
)
for name, comparison in comparisons.items():
    for plan, paired in comparison["plans"].items():
        print(
            f"{name:24s} {plan:14s} {paired['both']:5d} "
            f"{paired['reference_only']:9d} {paired['candidate_only']:8d} "
            f"{paired['difference']:+7.3f} {paired['p_value']:7.3g}"
        )

print("\nlocal_navigation regression check:")
print(f"{'skill':22s} " + " ".join(f"{name:>24s}" for name in benchmarks))
for index, result in enumerate(benchmarks["baseline"]["local_navigation"]):
    print(
        f"{result['skill_id']:22s} "
        + " ".join(
            f"{benchmark['local_navigation'][index]['success_rate']:24.3f}"
            for benchmark in benchmarks.values()
        )
    )

save_json(
    "comparisons",
    {
        name: {
            **comparison,
            "bands": {
                plan: {side: band_rows(table) for side, table in sides.items()}
                for plan, sides in comparison["bands"].items()
            },
        }
        for name, comparison in comparisons.items()
    },
)

In [ ]:
# Default-plan boundary success by skill and band, baseline against each run.
for name, comparison in comparisons.items():
    sides = comparison["bands"]["default"]
    reference, candidate = (
        {(row["skill"], row["band"]): row for row in band_rows(sides[side])}
        for side in ("reference", "candidate")
    )
    print(f"\n=== default plan: baseline against {name} ===")
    print(
        f"{'skill':22s} {'band':>11s} {'base_n':>6s} {'base':>6s} "
        f"{'run_n':>6s} {'run':>6s}"
    )
    for key in sorted(set(reference) | set(candidate)):
        old = reference.get(key, {"count": 0, "success_rate": float("nan")})
        new = candidate.get(key, {"count": 0, "success_rate": float("nan")})
        print(
            f"{key[0]:22s} {key[1]:>11s} {old['count']:6d} "
            f"{old['success_rate']:6.3f} {new['count']:6d} {new['success_rate']:6.3f}"
        )

## The success criterion, with the stochastic executor

Notebook 06 runs the executor stochastically, and that is where the 0.250 against 0.609 gap was measured. The greedy benchmark above is the one counterfactuals can replay, so the criterion is read again here in the setting the planner will actually use, on the same worlds. The starting worlds are paired, but the batch shares one key per step, so the executor's samples are not; the pairing is a lower bound on what it can buy.

In [ ]:
stochastic = {}
for name in executors:
    checkpoint = (
        BASELINE_CHECKPOINT if name == "baseline" else summaries[name]["checkpoint"]
    )
    stochastic[name] = benchmark_executor(
        learned_skill_executor_from_checkpoint(checkpoint, deterministic=False),
        num_episodes=preset["benchmark_episodes"],
        seed=BENCHMARK_SEED,
        skill_horizon=CHAIN_HORIZON,
        local_episodes_per_skill=0,
    )
    print_chains(name, stochastic[name])

print(
    f"\n{'run':24s} {'plan':14s} {'base_only':>9s} {'run_only':>8s} "
    f"{'diff':>7s} {'p':>7s}"
)
for name in summaries:
    for plan in stochastic[name]["chains"]:
        paired = compare_paired_successes(
            stochastic["baseline"]["chains"][plan],
            stochastic[name]["chains"][plan],
        )
        print(
            f"{name:24s} {plan:14s} {paired['reference_only']:9d} "
            f"{paired['candidate_only']:8d} {paired['difference']:+7.3f} "
            f"{paired['p_value']:7.3g}"
        )

# Across seeds, so a single lucky run cannot carry the conclusion.
randomized_runs = [name for name in summaries if name.startswith("randomized")]
if randomized_runs:
    for plan in ("default", "collect_first"):
        rates = [
            stochastic[name]["chains"][plan]["success_rate"]
            for name in randomized_runs
        ]
        print(
            f"randomized {plan:14s} mean={np.mean(rates):.3f} "
            f"min={min(rates):.3f} max={max(rates):.3f} seeds={len(rates)}"
        )
save_json(
    "stochastic-chains",
    {name: json_ready_benchmark(benchmark) for name, benchmark in stochastic.items()},
)

In [ ]:
names = list(benchmarks)
positions = np.arange(len(names))
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for axis, (title, source) in zip(
    axes,
    (("greedy", benchmarks), ("stochastic", stochastic)),
    strict=True,
):
    for offset, plan in ((-0.2, "default"), (0.2, "collect_first")):
        axis.bar(
            positions + offset,
            [source[name]["chains"][plan]["success_rate"] for name in names],
            width=0.4,
            label=plan,
        )
    axis.set_title(f"Chain success, {title} executor")
    axis.set_xticks(positions, names, rotation=30, ha="right")
    axis.set_ylim(0.0, 1.0)
axes[0].set_ylabel("Success rate")
axes[0].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Repeat the counterfactual probe on the best randomized run. If training
# worked for the reason Step 1 gave, the edit that rescued the most baseline
# boundaries should now rescue few.
if RUN_COUNTERFACTUALS and randomized_runs:
    best_run = max(
        randomized_runs,
        key=lambda name: stochastic[name]["chains"]["default"]["success_rate"],
    )
    best_boundaries = [
        boundary
        for boundaries in benchmarks[best_run]["boundaries"].values()
        for boundary in boundaries
    ]
    best_counterfactuals = probe_counterfactuals(
        executors[best_run],
        best_boundaries,
        skill_horizon=CHAIN_HORIZON,
    )
    print(
        f"{best_run}: boundaries={best_counterfactuals['num_boundaries']} "
        f"replay_agreement={best_counterfactuals['replay_agreement']:.3f}"
    )
    print(f"{'edit':20s} {'baseline diff':>13s} {'run diff':>9s} {'run p':>7s}")
    for name, comparison in best_counterfactuals["comparisons"].items():
        reference = counterfactuals["comparisons"].get(name, {})
        print(
            f"{name:20s} {reference.get('difference', float('nan')):+13.3f} "
            f"{comparison['difference']:+9.3f} {comparison['p_value']:7.3g}"
        )
    save_json(
        f"counterfactuals-{best_run}",
        {
            "num_boundaries": best_counterfactuals["num_boundaries"],
            "replay_agreement": best_counterfactuals["replay_agreement"],
            "comparisons": best_counterfactuals["comparisons"],
        },
    )
    print(f"\nBest run: {best_run}")
    print(f"Checkpoint for notebook 06: {summaries[best_run]['checkpoint']}")

## Next

- **Gap closed and no `local_navigation` regression:** point notebook 06's `CHECKPOINT` at the best run and rerun its `full` preset. That refits the competence prior for the new executor (its fits are saved beside the checkpoint) and reruns the planner ladder. The benchmark seed here is disjoint from notebook 06's seeds, so choosing the run here does not bias what notebook 06 reports.
- **`local-matched` gains as much as `randomized`:** the budget or the horizon did the work, not the scenario. Record that number, and keep the randomized scenario only if the counterfactuals show it removed a specific cost.
- **Gap unchanged:** read the knob that Step 1 blamed most, check that `randomized-eval` learned it, and turn that knob up or isolate it before training longer. Mined chain states from seeds disjoint from 7, 23, 51 and 101 are the next distribution to add. Prioritized replay comes after that, and only if success varies widely across bands.